# Modelado v3 — Solo 5 modelos de regresión
**David, Chiriquí 1940–2026**

Entrena Random Forest y XGBoost para:
- temperatura, humedad, lluvia, viento, presión

El índice de calor y nivel de alerta se calculan
matemáticamente — no son modelos separados.

## 1. Configuración

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

BASE_DIR = Path.cwd().parent

DATA_PROCESSED = BASE_DIR / "data" / "processed"

MODELS_DIR = BASE_DIR / "models"

REPORTS_DIR = BASE_DIR / "reports"

# Crear carpetas
MODELS_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Entorno configurado correctamente")

print(f"\nBASE_DIR:")
print(BASE_DIR)

print(f"\nDATA_PROCESSED:")
print(DATA_PROCESSED)

print(f"\nMODELS_DIR:")
print(MODELS_DIR)

print(f"\nREPORTS_DIR:")
print(REPORTS_DIR)

Entorno configurado correctamente

BASE_DIR:
c:\Users\Yajaira\Desktop\PROYECTO_PREDICCION_IA

DATA_PROCESSED:
c:\Users\Yajaira\Desktop\PROYECTO_PREDICCION_IA\data\processed

MODELS_DIR:
c:\Users\Yajaira\Desktop\PROYECTO_PREDICCION_IA\models

REPORTS_DIR:
c:\Users\Yajaira\Desktop\PROYECTO_PREDICCION_IA\reports


## 2. Carga del dataset

In [2]:
df = pd.read_csv(DATA_PROCESSED / 'dataset_features.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])
df = df.sort_values('Fecha').reset_index(drop=True)

print(f'Filas   : {len(df):,}')
print(f'Rango   : {df["Fecha"].min().date()} → {df["Fecha"].max().date()}')
print()
print('Targets:')
for t in [c for c in df.columns if c.startswith('Target_')]:
    print(f'  {t}')

Filas   : 13,260
Rango   : 1990-01-31 → 2026-05-21

Targets:
  Target_Temp_Max
  Target_Humedad
  Target_Lluvia
  Target_Viento_Max
  Target_Presion


In [3]:
print(df.columns.tolist())

['Fecha', 'Temp_Max', 'Temp_Min', 'Temp_Prom', 'Humedad', 'Lluvia', 'Lluvia_Hist', 'Presion', 'Viento_Max', 'Indice_Calor', 'Nivel_Alerta', 'Temp_Prom_lag1', 'Temp_Prom_lag7', 'Temp_Prom_lag30', 'Temp_Max_lag1', 'Temp_Max_lag7', 'IC_lag1', 'IC_lag7', 'Lluvia_lag1', 'Lluvia_lag7', 'Viento_lag1', 'Humedad_lag1', 'Humedad_lag7', 'Presion_lag1', 'Temp_Prom_m7', 'Temp_Prom_m30', 'Temp_Max_m7', 'IC_m7', 'IC_m30', 'Lluvia_m7', 'Lluvia_m30', 'Humedad_m7', 'Presion_m7', 'Dia_Año', 'Mes_Num', 'Estacion_Num', 'Target_Temp_Max', 'Target_Humedad', 'Target_Lluvia', 'Target_Viento_Max', 'Target_Presion']


## 3. Features y targets

In [4]:
features = [
    'Temp_Max', 'Temp_Min', 'Temp_Prom',
    'Humedad', 'Lluvia', 'Lluvia_Hist',
    'Presion', 'Viento_Max', 'Indice_Calor',
    'Temp_Prom_lag1', 'Temp_Prom_lag7', 'Temp_Prom_lag30',
    'Temp_Max_lag1',  'Temp_Max_lag7',
    'IC_lag1', 'IC_lag7',
    'Lluvia_lag1', 'Lluvia_lag7',
    'Viento_lag1',
    'Humedad_lag1', 'Humedad_lag7',
    'Presion_lag1',
    'Temp_Prom_m7',  'Temp_Prom_m30',
    'Temp_Max_m7',
    'IC_m7', 'IC_m30',
    'Lluvia_m7', 'Lluvia_m30',
    'Humedad_m7', 'Presion_m7',
    'Dia_Año', 'Mes_Num', 'Estacion_Num'
]

targets = {
    'temperatura': (df['Target_Temp_Max'],  '°C'),
    'humedad'    : (df['Target_Humedad'],    '%'),
    'lluvia'     : (df['Target_Lluvia'],     'mm'),
    'viento'     : (df['Target_Viento_Max'], 'km/h'),
    'presion'    : (df['Target_Presion'],    'hPa'),
}

X = df[features]
print(f'Features: {len(features)}')
print(f'Targets : {len(targets)}')

Features: 34
Targets : 5


## 4. División temporal (80% / 20%)

In [5]:
split = int(len(df) * 0.80)
X_train = X.iloc[:split]
X_test  = X.iloc[split:]

print(f'Entrenamiento : {len(X_train):,} filas  ({df["Fecha"].iloc[0].date()} → {df["Fecha"].iloc[split-1].date()})')
print(f'Prueba        : {len(X_test):,} filas  ({df["Fecha"].iloc[split].date()} → {df["Fecha"].iloc[-1].date()})')

Entrenamiento : 10,608 filas  (1990-01-31 → 2019-02-15)
Prueba        : 2,652 filas  (2019-02-16 → 2026-05-21)


## 5. Entrenar Random Forest

In [ ]:
print('Entrenando Random Forest...')
print()
resultados_rf = {}

for nombre, (y, unidad) in targets.items():
    y_train = y.iloc[:split]
    y_test  = y.iloc[split:]
    modelo  = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    modelo.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, modelo.predict(X_test))
    joblib.dump(modelo, MODELS_DIR / f'om_rf_{nombre}.pkl')
    resultados_rf[nombre] = (mae, unidad)
    print(f' {nombre:12s} → MAE: {mae:.2f} {unidad}')

print()
print('Random Forest completado.')

Entrenando Random Forest...

  ✓ temperatura  → MAE: 1.02 °C
  ✓ humedad      → MAE: 2.65 %
  ✓ lluvia       → MAE: 8.15 mm
  ✓ viento       → MAE: 1.89 km/h
  ✓ presion      → MAE: 0.51 hPa

Random Forest completado.


## Metricas

In [10]:
print('Entrenando Random Forest...')
print()

resultados_rf = {}

for nombre, (y, unidad) in targets.items():
    y_train = y.iloc[:split]
    y_test = y.iloc[split:]

    # ============================================
    # MODELO
    # ============================================

    modelo = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(X_train, y_train)

    # ============================================
    # PREDICCIONES
    # ============================================

    pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    r2 = r2_score(y_test, pred)

    joblib.dump(
        modelo,
        MODELS_DIR / f'om_rf_{nombre}.pkl'
    )

    resultados_rf[nombre] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

    print(f'{nombre}')
    print(f'   MAE  : {mae:.2f} {unidad}')
    print(f'   RMSE : {rmse:.2f} {unidad}')
    print(f'   R²   : {r2:.4f}')
    print('-' * 40)

print()

print('Random Forest completado.')

Entrenando Random Forest...

temperatura
   MAE  : 1.02 °C
   RMSE : 1.30 °C
   R²   : 0.7008
----------------------------------------
humedad
   MAE  : 2.65 %
   RMSE : 3.77 %
   R²   : 0.8334
----------------------------------------
lluvia
   MAE  : 8.15 mm
   RMSE : 11.63 mm
   R²   : 0.1441
----------------------------------------
viento
   MAE  : 1.89 km/h
   RMSE : 2.68 km/h
   R²   : 0.1870
----------------------------------------
presion
   MAE  : 0.51 hPa
   RMSE : 0.64 hPa
   R²   : 0.7293
----------------------------------------

Random Forest completado.


Interpretación temperatura

El modelo logra predecir la temperatura con un error promedio cercano a 1 °C, lo cual representa un resultado bastante bueno para datos climáticos reales.
El valor de R² indica que el modelo explica aproximadamente el 70% de la variabilidad de la temperatura.

Interpretación Humedad

La humedad fue una de las variables con mejor desempeño.
El modelo presenta errores bajos y un R² de 0.83, indicando que explica más del 83% del comportamiento de la humedad relativa.

Interpretación Luvia

La predicción de lluvia obtuvo el rendimiento más bajo.
Esto ocurre porque la precipitación es una variable altamente variable y difícil de modelar debido a cambios bruscos y eventos extremos.
El modelo solo logra explicar aproximadamente el 14% de la variabilidad de la lluvia.

Interpretación Viento

El modelo presenta un desempeño moderado para la velocidad del viento.
Aunque el error absoluto es relativamente bajo, el R² indica que la capacidad explicativa del modelo todavía es limitada para esta variable.

## 6. Entrenar XGBoost

In [11]:
print('Entrenando XGBoost...')
print()
resultados_xgb = {}

params = dict(n_estimators=300, learning_rate=0.05, max_depth=5,
              subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)

for nombre, (y, unidad) in targets.items():
    y_train = y.iloc[:split]
    y_test  = y.iloc[split:]
    modelo  = XGBRegressor(**params)
    modelo.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, modelo.predict(X_test))
    joblib.dump(modelo, MODELS_DIR / f'om_xgb_{nombre}.pkl')
    resultados_xgb[nombre] = (mae, unidad)
    print(f'  {nombre:12s} → MAE: {mae:.2f} {unidad}')

print()
print('XGBoost completado.')

Entrenando XGBoost...

  temperatura  → MAE: 0.97 °C
  humedad      → MAE: 2.59 %
  lluvia       → MAE: 7.18 mm
  viento       → MAE: 1.89 km/h
  presion      → MAE: 0.50 hPa

XGBoost completado.


In [12]:
print('Entrenando XGBoost...')
print()

resultados_xgb = {}
params = dict(

    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

for nombre, (y, unidad) in targets.items():

    y_train = y.iloc[:split]
    y_test = y.iloc[split:]

    # ============================================
    # MODELO
    # ============================================

    modelo = XGBRegressor(**params)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    # ============================================
    # MÉTRICAS
    # ============================================
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    r2 = r2_score(y_test, pred)

    joblib.dump(
        modelo,

        MODELS_DIR / f'om_xgb_{nombre}.pkl'
    )

    resultados_xgb[nombre] = {

        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

    print(f'{nombre}')
    print(f'   MAE  : {mae:.2f} {unidad}')
    print(f'   RMSE : {rmse:.2f} {unidad}')
    print(f'   R²   : {r2:.4f}')
    print('-' * 40)

print()

print('XGBoost completado.')

Entrenando XGBoost...

temperatura
   MAE  : 0.97 °C
   RMSE : 1.24 °C
   R²   : 0.7291
----------------------------------------
humedad
   MAE  : 2.59 %
   RMSE : 3.68 %
   R²   : 0.8411
----------------------------------------
lluvia
   MAE  : 7.18 mm
   RMSE : 10.70 mm
   R²   : 0.2763
----------------------------------------
viento
   MAE  : 1.89 km/h
   RMSE : 2.63 km/h
   R²   : 0.2168
----------------------------------------
presion
   MAE  : 0.50 hPa
   RMSE : 0.63 hPa
   R²   : 0.7401
----------------------------------------

XGBoost completado.


Interpretación Temperatura: 
El modelo logró predecir la temperatura con un error promedio menor a 1 °C, lo cual representa una precisión bastante alta para predicción climática.
El valor de R² indica que el modelo explica aproximadamente el 73% de la variabilidad de la temperatura.

Interpretación Humedad: 
La humedad fue una de las variables con mejor desempeño.
El modelo presenta errores bajos y un R² superior al 84%, demostrando una excelente capacidad predictiva para esta variable.

Interpretación Lluvia: 
Aunque la lluvia continúa siendo la variable más difícil de modelar, XGBoost logró mejorar considerablemente respecto a Random Forest.
El modelo consiguió explicar aproximadamente el 28% de la variabilidad de la precipitación.

Interpretación Viento: 
La velocidad del viento presentó un rendimiento moderado.
Aunque el error absoluto es relativamente bajo, el valor de R² indica que todavía existe dificultad para modelar completamente el comportamiento del viento.

## 7. Comparación RF vs XGBoost

In [14]:
print('=' * 70)

print('      COMPARACIÓN — Random Forest vs XGBoost')
print('=' * 70)

print(
    f'{"Modelo":12s}'
    f'{"RF MAE":>12s}'
    f'{"XGB MAE":>12s}'
    f'{"RF R²":>12s}'
    f'{"XGB R²":>12s}'
    f'{"Ganador":>12s}'
)

print('-' * 70)

for nombre in targets:

    mae_rf = resultados_rf[nombre]['MAE']
    r2_rf = resultados_rf[nombre]['R2']
    mae_xgb = resultados_xgb[nombre]['MAE']
    r2_xgb = resultados_xgb[nombre]['R2']

    ganador = 'XGBoost' if mae_xgb < mae_rf else 'RF'

    print(
        f'{nombre:12s}'
        f'{mae_rf:12.2f}'
        f'{mae_xgb:12.2f}'
        f'{r2_rf:12.4f}'
        f'{r2_xgb:12.4f}'
        f'{ganador:>12s}'
    )

print('=' * 70)

print()

print('Nota:')
print('XGBoost mostró mejor desempeño general en la mayoría')
print('de variables climáticas analizadas.')

      COMPARACIÓN — Random Forest vs XGBoost
Modelo            RF MAE     XGB MAE       RF R²      XGB R²     Ganador
----------------------------------------------------------------------
temperatura         1.02        0.97      0.7008      0.7291     XGBoost
humedad             2.65        2.59      0.8334      0.8411     XGBoost
lluvia              8.15        7.18      0.1441      0.2763     XGBoost
viento              1.89        1.89      0.1870      0.2168     XGBoost
presion             0.51        0.50      0.7293      0.7401     XGBoost

Nota:
XGBoost mostró mejor desempeño general en la mayoría
de variables climáticas analizadas.


## 8. Validación de consistencia — prueba con datos reales

In [15]:
# Tomar el último día del conjunto de prueba
muestra = X_test.iloc[[-1]]
fecha_real = df['Fecha'].iloc[split + len(X_test) - 1].date()

# Predicciones XGBoost
pred_temp    = round(float(joblib.load(MODELS_DIR / 'om_xgb_temperatura.pkl').predict(muestra)[0]), 2)
pred_humedad = round(float(joblib.load(MODELS_DIR / 'om_xgb_humedad.pkl').predict(muestra)[0]),    1)
pred_lluvia  = round(max(0.0, float(joblib.load(MODELS_DIR / 'om_xgb_lluvia.pkl').predict(muestra)[0])), 1)
pred_viento  = round(float(joblib.load(MODELS_DIR / 'om_xgb_viento.pkl').predict(muestra)[0]),    1)
pred_presion = round(float(joblib.load(MODELS_DIR / 'om_xgb_presion.pkl').predict(muestra)[0]),   1)

# Calcular índice de calor con temperatura y humedad predichas
T, H = pred_temp, pred_humedad
ic_calculado = round(
    -8.78469475556 + 1.61139411*T + 2.33854883889*H - 0.14611605*T*H
    - 0.012308094*T**2 - 0.0164248277778*H**2 + 0.002211732*T**2*H
    + 0.00072546*T*H**2 - 0.000003582*T**2*H**2, 1
)

# Nivel de alerta por tabla OMS
if ic_calculado >= 41: alerta = 'PELIGRO EXTREMO'
elif ic_calculado >= 35: alerta = 'PELIGRO'
elif ic_calculado >= 32: alerta = 'PRECAUCION'
else: alerta = 'NORMAL'

print(f'Validación con datos del {fecha_real}:')
print(f'  Temperatura predicha  : {pred_temp}°C')
print(f'  Humedad predicha      : {pred_humedad}%')
print(f'  Lluvia predicha       : {pred_lluvia} mm')
print(f'  Viento predicho       : {pred_viento} km/h')
print(f'  Presión predicha      : {pred_presion} hPa')
print()
print(f'  Índice de calor (calc): {ic_calculado}°C  ← calculado con temp+humedad predichos')
print(f'  Nivel de alerta (OMS) : {alerta}  ← 100% consistente con el índice')

Validación con datos del 2026-05-21:
  Temperatura predicha  : 31.8°C
  Humedad predicha      : 88.9%
  Lluvia predicha       : 12.0 mm
  Viento predicho       : 9.5 km/h
  Presión predicha      : 1004.5 hPa

  Índice de calor (calc): 47.6°C  ← calculado con temp+humedad predichos
  Nivel de alerta (OMS) : PELIGRO EXTREMO  ← 100% consistente con el índice
